# Fine Tuning Llama 2 Model

## 01 Install required packages

In [ ]:
%pip install -q accelerate bitsandbytes peft transformers trl

## 02 Import required packages

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
  AutoModelForCausalLM,
  AutoTokenizer,
  BitsAndBytesConfig,
  HfArgumentParser,
  TrainingArguments,
  pipeline,
  logging
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

## 03 Define Parameters

1. Load a llama-2-7b-chat-hf-model
2. Train in to the `mlabonne/guanaco-llama2-1k` which wil produce our fiine tuned model

In [ ]:
# Model
model_name = "NousResearch/Llama-2-7b-chat-hf"

# dataset
dataset_name = "mlabonne/guanaco-llama2-1k"

# fine-tuned model name
fine_tuned_model_name = "Llama-2-7b-chat-finetune"

### QLoRa Parameters

In [ ]:
# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

### bitsandbytes parameters

In [ ]:
# Activate 4-bit precision base model loading
use_4bit = True

# compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double optimization)
use_nested_quant = False

### TrainingArguments parameters

In [ ]:
output_dir = "./results"

# number of training epochs
epochs = 1

# Enable fp16/bf16 training
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal
max_grad_norm = 0.3

# Initial lerning rate
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_schedule_type = "cosine"

# Number of training steps
max_steps = -1

# Ratio of steps for linear warmup
warmup_ratio = 0.03

# Group sequences into batches with same length
group_by_length = True

# Save checkpoint every X update steps
save_steps = 0

# Loggin steps
loggin_steps = 25

### SFT parameters

In [ ]:
# Maximum sequence length
max_seq_length = None

# Pack multiple short examples in the same input sequence 
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

## 04. Fine tuning process

1. Load the dataset
2. Preprocess the dataset to reformat to the prompt structure.
3. Configure `bitsandbytes` for 4-bit quantization.
4. Load Llama 2 model in 4-bit precision on a GPU with the corresponding tokens.
5. Load configurations for QLoRA, regular training parameters and passing everything to SFTTrainer.

In [ ]:
# Load dataset
dataset = load_dataset(dataset_name, split="train")

# Load tokenizer
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
  load_in_4bit=use_4bit,
  bnb_4bit_quant_type=bnb_4bit_quant_type,
  bnb_4bit_compute_dtype=compute_dtype,
  bnb_4bit_use_double_quant=use_nested_quant
)

In [ ]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map="auto"
)

In [ ]:
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=loggin_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_schedule_type,
    report_to="tensorboard",
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)


In [ ]:
# train the mode
trainer.train(model)

In [ ]:
# save trained model

trainer.model.save_pretrained(fine_tuned_model_name)

In [ ]:
 # ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = "What is a large language model?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]\n")
print(result[0]['generated_text'])